# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaivarshini-06/flyrank-ml-internship-jaivarshini/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*
## Research Question

Can historical content-performance signals help prioritize which content items should be reviewed for a potential refresh?

## Decision Supported

The system produces a ranked list of content items for human review. An editor can use the ranking to decide which items to investigate first, rather than treating the score as an instruction to refresh content automatically.

## Project Objective

Develop a transparent baseline and compare it with a machine learning model using an appropriate evaluation split. The goal is to assess whether the model provides useful prioritization beyond the baseline, while avoiding data leakage and documenting limitations.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data Source and Preparation

### Dataset Release
This project uses the FlyRank Internship — Pseudonymized Warehouse Release (v20260703), a frozen warehouse snapshot exported on July 3, 2026.

### Tables Used
- `dim_content`: content metadata, including content type, creation and update dates, word count, search volume, and publication status.
- `fact_content_daily_performance`: daily performance metrics, including impressions, clicks, average search position, pageviews, and sessions.

The warehouse provides monthly performance partitions from January 2025 through June 2026. The sample file was used for initial schema inspection.

### Date Windows
The final training, validation, and test windows will be specified in the methodology and evaluation sections. They will be selected to preserve temporal order and avoid using future information as a model input.

### Exclusions and Data Safety
Pseudonymous client, content, keyword, and URL identifiers will not be used as predictive features. Client-identifying information, URLs, and private queries will not appear in public outputs. Label-derived fields, including `trend_direction` and `trend_pct`, will be excluded from model inputs. Features will be restricted to information available at the prediction time, and unavailable performance data will not automatically be treated as zero performance..

In [1]:
%pip -q install duckdb huggingface_hub pandas pyarrow

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
from huggingface_hub import HfApi
api = HfApi()
try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("Dataset metadata is accessible!")
    print("Dataset ID:", info.id)
    print("Private/gated:", info.private, info.gated)
except Exception as e:
    print("Dataset access check failed.")
    print("Error:", str(e)[:500])

Dataset metadata is accessible!
Dataset ID: FlyRank/internship-warehouse
Private/gated: False auto


In [4]:
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)
print("Available dataset files:\n")
for file in files:
    print(file)

Available dataset files:

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_cont

In [6]:
import duckdb
import getpass
hf_token = getpass.getpass("Enter your Hugging Face read token: ")
con = duckdb.connect()
safe_token = hf_token.replace("'", "''")
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{safe_token}')"
)

sample_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance_sample.parquet"
)

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{sample_path}')"
).df()

print("Sample file columns and data types:")
display(schema)

print("\nSample rows:")
display(
    con.execute(
        f"SELECT * FROM read_parquet('{sample_path}') LIMIT 5"
    ).df()
)

Enter your Hugging Face read token: ··········


HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance_sample.parquet' (HTTP 401)

In [7]:
from huggingface_hub import HfApi
from huggingface_hub.utils import get_token
token = get_token()

if not token:
    print("No saved token found. We need to log in again.")
else:
    try:
        api = HfApi(token=token)
        user = api.whoami()
        print("Hugging Face login works.")
        print("Authenticated account:", user.get("name", "verified"))

        info = api.get_paths_info(
            repo_id="FlyRank/internship-warehouse",
            paths=["fact_content_daily_performance_sample.parquet"],
            repo_type="dataset",
        )
        print("Sample file metadata accessible:", bool(info))

    except Exception as e:
        print("Access check failed:", str(e)[:300])

Hugging Face login works.
Authenticated account: Jaivarshini-06
Sample file metadata accessible: True


In [8]:
from huggingface_hub import hf_hub_download, get_token
import pandas as pd

print("Downloading the sample file...")

sample_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=get_token()
)

print("Sample file ready!")

# Inspect the schema
sample_schema = pd.read_parquet(sample_file).head(5)
print("\nColumns:")
print(sample_schema.columns.tolist())

print("\nData types:")
print(sample_schema.dtypes)

print("\nFirst 5 rows:")
display(sample_schema)

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Sample file ready!

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Data types:
report_date                  object
client_hash_id               object
content_hash_id              object
client_has_gsc                 bool
client_has_ga4                 bool
gsc_data_available             bool
ga4_data_available           object
gsc_impressions               int64
gsc_clicks                    int64
gsc_sum_position            float64
gsc_avg_position            float64
ga4_pageviews              

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06


In [9]:
from huggingface_hub import hf_hub_download, get_token
import pandas as pd

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=get_token()
)

content = pd.read_parquet(content_file)

print("Shape:", content.shape)
print("\nColumns:")
print(content.dtypes.to_string())
print("\nFirst 3 rows:")
display(content.head(3))

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Shape: (519606, 26)

Columns:
client_hash_id                 object
content_hash_id                object
keyword_hash_id                object
url_hash_id                    object
keyword_char_count              int64
keyword_token_count             int64
url_char_count                  int64
content_created_date           object
content_updated_date           object
content_type                   object
search_volume                 float64
competition                   float64
competition_level              object
cpc                           float64
main_intent                    object
backlinks                     float64
category_count                  int64
keyword_created_date           object
provider_used                  object
model_used                     object
char_count                    float64
word_count                    float64
last_optimized_date            object
optimization_eligible_date     object
is_published                     bool
is_deleted          

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
